# Sample Eval Prompts/Responses Across Models

Pick a set of finetuned experiments + base-model baselines, draw `n=20` random `(prompt, response)` pairs from each model's cached generation-eval responses, and save them as paper-pasteable `.md` / `.tex` / `.csv` / `.json`.

Companion to `explore_models.ipynb` (model filtering) and `view_results_v2.ipynb` (aggregate metrics) — this one is for showing **what the models actually said**.

Workflow: edit the filter cells → run all → the saved bundle lands in `notebooks/samples/<RUN_NAME>/`.

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import random
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from loguru import logger

from sl import config as sl_config
from sl.results import (
    build_baseline_df,
    build_gen_df,
    filter_gen_df,
    load_registry,
)
from sl.tables import savetable

ARTIFACTS_DIR = Path(sl_config.ARTIFACTS_DIR)
SAMPLES_DIR = Path.cwd() / "samples"

reg = load_registry()
gen_df = build_gen_df(reg)
baseline_df = build_baseline_df(reg)

logger.info(
    f"gen_df: {len(gen_df)} rows; baseline_df: {len(baseline_df)} rows; "
    f"animals={sorted(gen_df['animal'].dropna().unique())}"
)

## 1. Filter finetuned experiments

Same knobs as `filter_gen_df`. Set any knob to `None` to skip it; for prompt columns use `"<none>"` to match nulls and `""` to match the explicit empty string (mirrors `explore_models.ipynb`).

In [ ]:
ft_filtered = filter_gen_df(
    gen_df,
    animals="cat",
    eval_setting="clean",
    dwg_mode="full",
    svd_mode="full",
    full_ft=False,
)
ft_filtered = ft_filtered.sort_values("p_target", ascending=False).reset_index(drop=True)

print(f"{len(ft_filtered)} matching (experiment, eval_setting) rows")
ft_filtered.head(15)[
    [
        "exp_id",
        "model_hash",
        "animal",
        "variant",
        "rank",
        "epochs",
        "training_seed",
        "eval_setting",
        "p_target",
    ]
]

## 2. Choose which models to include

- `EXPLICIT_FT_HASHES` — set to a list of `model_hash`es (and matching `eval_setting`s) for full control. When `None`, the top `N_FT_MODELS` rows from `ft_filtered` are used.
- `BASELINE_ANIMALS` / `BASELINE_BASE_MODELS` — pick base-model baselines (from `baseline_df`) for comparison. Set to `[]` to skip baselines.

In [ ]:
N_FT_MODELS = 3
EXPLICIT_FT_HASHES: list[str] | None = None
EXPLICIT_FT_EVAL_SETTING: str | None = None

BASELINE_ANIMALS = ["cat"]
BASELINE_BASE_MODELS: list[str] | None = None

if EXPLICIT_FT_HASHES is not None:
    candidates = gen_df[gen_df["model_hash"].isin(EXPLICIT_FT_HASHES)]
    if EXPLICIT_FT_EVAL_SETTING is not None:
        candidates = candidates[candidates["eval_setting"] == EXPLICIT_FT_EVAL_SETTING]
    ft_selected = (
        candidates.sort_values("model_hash")
        .drop_duplicates("model_hash", keep="first")
        .reset_index(drop=True)
    )
    missing = sorted(set(EXPLICIT_FT_HASHES) - set(ft_selected["model_hash"]))
    if missing:
        logger.warning(f"No registry rows found for hashes: {missing}")
else:
    ft_selected = ft_filtered.head(N_FT_MODELS).reset_index(drop=True)

baseline_selected = baseline_df.copy()
if BASELINE_ANIMALS is not None:
    baseline_selected = baseline_selected[baseline_selected["animal"].isin(BASELINE_ANIMALS)]
if BASELINE_BASE_MODELS:
    base_mask = pd.Series(False, index=baseline_selected.index)
    for bm in BASELINE_BASE_MODELS:
        base_mask |= (
            baseline_selected["base_model"].fillna("").str.contains(bm, case=False, regex=False)
        )
    baseline_selected = baseline_selected[base_mask]
baseline_selected = baseline_selected.reset_index(drop=True)

print(f"Finetuned models: {len(ft_selected)}")
display(
    ft_selected[
        [
            "exp_id",
            "model_hash",
            "animal",
            "variant",
            "rank",
            "eval_setting",
            "p_target",
        ]
    ]
)
print(f"Baseline models: {len(baseline_selected)}")
display(baseline_selected[["baseline_key", "animal", "base_model", "n_responses", "p_target"]])

## 3. Draw `n` random (prompt, response) pairs per model

Sampling is **without replacement** within a model and seeded for reproducibility. A model's responses file is the same file `view_results_v2`/`paper_figures` use for aggregate metrics, so these samples represent the same eval distribution.

In [ ]:
N_SAMPLES_PER_MODEL = 20
SEED = 42

_LEGACY_PREFIX = "/net/projects/clab/subliminal/shared/results/"


def _resolve_responses_path(path_str: str) -> Path | None:
    """Older registry rows reference the legacy `/net/projects/clab/...` mount;
    current writes go to `<ARTIFACTS_DIR>`. Try as-recorded first, then rewrite."""
    candidates = [Path(path_str)]
    if _LEGACY_PREFIX in path_str:
        candidates.append(Path(path_str.replace(_LEGACY_PREFIX, str(ARTIFACTS_DIR) + "/")))
    candidates.append(
        ARTIFACTS_DIR / "responses" / Path(path_str).parent.name / Path(path_str).name
    )
    for c in candidates:
        if c.exists():
            return c
    return None


def _sample_pairs(prompts_data: list[dict], n: int, rng: random.Random) -> list[tuple[str, str]]:
    """Flatten `[{prompt, responses: [...]}]` to (prompt, response) pairs and
    sample `n` without replacement. Returns `min(n, total)` pairs."""
    pairs = [
        (block.get("prompt", ""), resp)
        for block in prompts_data
        for resp in block.get("responses", [])
    ]
    if not pairs:
        return []
    return rng.sample(pairs, min(n, len(pairs)))


def _short_model(name: str | None) -> str:
    return name.rsplit("/", 1)[-1] if name else "?"


def _ft_label(row: pd.Series) -> str:
    rank = "" if pd.isna(row.get("rank")) else f" r{int(row['rank'])}"
    setting = f" [{row['eval_setting']}]" if row.get("eval_setting") else ""
    return f"{row['animal']}{rank}{setting} ({row['model_hash']})"


rng = random.Random(SEED)
rows: list[dict] = []

for _, row in ft_selected.iterrows():
    exp = reg.get("experiments", {}).get(row["exp_id"], {})
    resp_paths = (exp.get("results") or {}).get("responses_paths") or {}
    raw_path = resp_paths.get(row["eval_setting"])
    if not raw_path:
        logger.warning(f"No responses_paths[{row['eval_setting']}] for {row['exp_id']}")
        continue
    path = _resolve_responses_path(raw_path)
    if path is None:
        logger.warning(f"Responses file missing for {row['exp_id']}: {raw_path}")
        continue
    with open(path) as f:
        data = json.load(f)
    label = _ft_label(row)
    for prompt, response in _sample_pairs(data, N_SAMPLES_PER_MODEL, rng):
        rows.append(
            {
                "kind": "finetuned",
                "model_label": label,
                "model_id": row["model_hash"],
                "exp_id": row["exp_id"],
                "animal": row["animal"],
                "variant": row.get("variant"),
                "rank": (None if pd.isna(row.get("rank")) else int(row["rank"])),
                "eval_setting": row["eval_setting"],
                "prompt": prompt,
                "response": response,
            }
        )

for _, brow in baseline_selected.iterrows():
    entry = reg.get("baselines", {}).get(brow["baseline_key"], {})
    gr = (entry.get("generation_results") or {}).get("clean") or []
    raw_data: list[dict] = []
    seen_paths: set[str] = set()
    for r in gr:
        rp = r.get("responses_path")
        if not rp or rp in seen_paths:
            continue
        seen_paths.add(rp)
        path = _resolve_responses_path(rp)
        if path is None:
            logger.warning(f"Baseline responses missing for {brow['baseline_key']}: {rp}")
            continue
        with open(path) as f:
            raw_data.extend(json.load(f))
    if not raw_data:
        logger.warning(f"No usable responses for baseline {brow['baseline_key']}")
        continue
    label = f"{_short_model(brow['base_model'])} no-FT [{brow['animal']}] ({brow['baseline_key']})"
    for prompt, response in _sample_pairs(raw_data, N_SAMPLES_PER_MODEL, rng):
        rows.append(
            {
                "kind": "baseline",
                "model_label": label,
                "model_id": brow["baseline_key"],
                "exp_id": None,
                "animal": brow["animal"],
                "variant": "baseline",
                "rank": None,
                "eval_setting": "clean",
                "prompt": prompt,
                "response": response,
            }
        )

samples_df = pd.DataFrame(rows)
logger.success(
    f"Sampled {len(samples_df)} (prompt, response) pairs across "
    f"{samples_df['model_label'].nunique() if not samples_df.empty else 0} models "
    f"(n={N_SAMPLES_PER_MODEL} each, seed={SEED})"
)
samples_df.head()

## 4. View per-model tables

In [ ]:
from IPython.display import Markdown, display

with pd.option_context("display.max_colwidth", 240):
    for label, sub in samples_df.groupby("model_label", sort=False):
        display(Markdown(f"### {label}  \n_n={len(sub)} pairs_"))
        display(sub[["prompt", "response"]].reset_index(drop=True))

## 5. Save to disk

Bundle written under `notebooks/samples/<RUN_NAME>/`:

- `samples.json` — long-form records, one per pair.
- `samples.csv` — same data, spreadsheet-friendly.
- `samples.md` — per-model Markdown tables (drop straight into a write-up).
- `samples_<RUN_NAME>.tex` + `.csv` — paper-ready LaTeX via `sl.tables.savetable` (handles `%`/Unicode escaping).

In [ ]:
RUN_NAME = "cat_pilot"

run_dir = SAMPLES_DIR / RUN_NAME
run_dir.mkdir(parents=True, exist_ok=True)

samples_df.to_csv(run_dir / "samples.csv", index=False)
samples_df.to_json(run_dir / "samples.json", orient="records", indent=2)


def _md_cell(s: str) -> str:
    """Inline-safe Markdown table cell: collapse newlines and escape pipes."""
    return (s or "").replace("\\", "\\\\").replace("|", "\\|").replace("\n", " ")


md_lines = [
    f"# Random eval samples \u2014 {RUN_NAME}",
    "",
    f"_n={N_SAMPLES_PER_MODEL} per model, seed={SEED}, total pairs={len(samples_df)}_",
    "",
]
for label, sub in samples_df.groupby("model_label", sort=False):
    md_lines += [f"## {label}", "", "| Prompt | Response |", "| --- | --- |"]
    md_lines += [f"| {_md_cell(r.prompt)} | {_md_cell(r.response)} |" for r in sub.itertuples()]
    md_lines.append("")
(run_dir / "samples.md").write_text("\n".join(md_lines) + "\n")

latex_df = samples_df[["model_label", "prompt", "response"]].rename(
    columns={"model_label": "Model", "prompt": "Prompt", "response": "Response"}
)
savetable(
    latex_df,
    name=f"samples_{RUN_NAME}",
    out_dir=run_dir,
    caption=(
        f"Random sample of {N_SAMPLES_PER_MODEL} "
        f"(prompt, response) pairs per model (seed={SEED})."
    ),
    label=f"tab:samples_{RUN_NAME}",
    index=False,
)

logger.success(f"Wrote bundle to {run_dir}")
sorted(p.name for p in run_dir.iterdir())